### End to End Parsing and Eval: Baselines

In [5]:
import csv,re, pandas as pd
import numpy as np
from io import StringIO
def extract_final_table_as_csv(file_contents: str) -> pd.DataFrame:
    """
    1. Finds the section after '### Final Table:'.
    2. Extracts all lines starting with '|' and ending with '| <NEWLINE>'.
    3. Splits each line by '|' to get individual cells.
    4. Builds a pandas DataFrame from the parsed rows (first row becomes column headers).
    """
    # 1. Locate the "### Final Table:" section up to next '###' or end of file
    table_match = re.search(r'###\s*Final\s*Table:(.*?)(?=\n###|$)', file_contents, flags=re.DOTALL)
    if not table_match:
        raise ValueError("No '### Final Table:' section found in the file text.")
    
    table_section = table_match.group(1).strip()
    
    # 2. Extract lines that start with '|' and end with '| <NEWLINE>'
    lines = re.findall(r'^\|(.*?)\|\s*<NEWLINE>', table_section, flags=re.MULTILINE)
    if not lines:
        raise ValueError("No table lines found in the final table section.")

    # 3. Parse each line into cells by splitting on '|'
    table_data = []
    for line in lines:
        # Split on '|', strip spaces
        cells = [cell.strip() for cell in line.split('|')]
        table_data.append(cells)

    # 4. Build the DataFrame:
    #    - The first row is assumed to be the header.
    #    - Subsequent rows are data.
    header = table_data[0]
    data_rows = table_data[1:]
    
    df = pd.DataFrame(data_rows, columns=header)

    # Optional: If you want to convert numeric columns to actual numbers,
    # you can attempt a conversion for each column except the first (which might be 'Team'):
    # for col in header[1:]:
    #     df[col] = pd.to_numeric(df[col], errors='ignore')

    return df


def compute_rmse_and_er_cell_by_cell_baseline(df_true: pd.DataFrame, df_pred: pd.DataFrame) -> (float, float):
    """
    Computes the RMSE and Error Rate between two DataFrames that each contain rows
    for "Home Team" and "Away Team" and matching numeric columns. In this version,
    df_pred is cleaned by stripping leading/trailing whitespace from all cells
    (including row/index labels and column labels).
    """
    # 1. Strip leading/trailing spaces from column labels in df_pred
    df_pred.columns = [col.strip() for col in df_pred.columns]
    
    # 2. If df_pred has 'Team' as a column, strip its values as well before setting it as index
    if 'Team' in df_pred.columns:
        df_pred['Team'] = df_pred['Team'].str.strip()
        
    # 3. Strip leading/trailing spaces from index labels if df_pred is already indexed
    df_pred.index = [str(idx).strip() for idx in df_pred.index]
    
    # 4. Strip spaces from every cell in df_pred (if the cell is a string)
    df_pred = df_pred.applymap(lambda x: x.strip() if isinstance(x, str) else x)
    
    # 5. Rename 'Shots Taken' to 'Shots' in df_pred if present (now that spaces are stripped)
    if 'Shots Taken' in df_pred.columns:
        df_pred.rename(columns={'Shots Taken': 'Shots'}, inplace=True)
    
    # 6. Process df_true: set 'Team' as the index if it exists, remove the index name
    if 'Team' in df_true.columns:
        df_true.set_index('Team', inplace=True)
    df_true.index.name = None
    
    # 7. df_pred: likewise, set 'Team' as the index if it exists, remove index name
    if 'Team' in df_pred.columns:
        df_pred.set_index('Team', inplace=True)
    df_pred.index.name = None

    # The expected rows in a fixed order (no leading/trailing spaces)
    expected_rows = ["Home Team", "Away Team"]
    
    # 8. Ensure df_true contains the two teams
    if not set(expected_rows).issubset(df_true.index):
        raise ValueError("The gold table does not contain both 'Home Team' and 'Away Team'.")
    
    # 9. Ensure df_pred contains the two teams (after trimming)
    if not set(expected_rows).issubset(df_pred.index):
        raise ValueError("The predicted table does not contain both 'Home Team' and 'Away Team'.")
    
    # 10. Reorder rows in both DataFrames to match [Home Team, Away Team]
    df_true = df_true.loc[expected_rows]
    df_pred = df_pred.loc[expected_rows]
    
    # 11. Align df_pred to df_true's columns for one-to-one cell mapping
    df_pred = df_pred.reindex(columns=df_true.columns)

    # 12. Check shapes after alignment
    if df_true.shape != df_pred.shape:
        raise ValueError("Both tables must have the same shape after alignment.")

    # 13. Ensure that both have the same columns
    if list(df_true.columns) != list(df_pred.columns):
        raise ValueError("Both tables must have the same columns in the same order.")

    # 14. Number of cells
    n = df_true.size
    if n == 0:
        raise ValueError("Tables are empty; no cells to compare.")
    
    # 15. Extract values as numpy arrays
    true_values = df_true.values
    pred_values = df_pred.values
    
    squared_diff_sum = 0.0
    error_count = 0

    # 16. Iterate cell-by-cell
    for r in range(df_true.shape[0]):
        for c in range(df_true.shape[1]):
            gt_val = true_values[r, c]
            pred_val = pred_values[r, c]
            
            # Try to convert to float if they are string
            # (This is useful if your columns are supposed to be numeric but
            #  ended up as strings after trimming.)
            if isinstance(gt_val, str) and gt_val.isdigit():
                gt_val = float(gt_val)
            if isinstance(pred_val, str) and pred_val.isdigit():
                pred_val = float(pred_val)
            
            # Check cell difference for RMSE
            # Will raise a TypeError if they're not numeric (float or int).
            diff = gt_val - pred_val
            squared_diff_sum += diff * diff
            
            # Check exact match for Error Rate
            if gt_val != pred_val:
                error_count += 1

    # 17. Compute metrics
    rmse = np.sqrt(squared_diff_sum / n)
    # Multiply by 100 if you want the Error Rate in percentage
    er = (error_count / n) * 100

    return rmse, er

path = 'data/livesum/test.json'
data = pd.read_json(path)
outputs = []
gold = []
# Example usage:
for idx in range(0,100):
    fname = f"model_outputs/Livesum/GPT4o_Baseline/{idx}.txt"
    with open(fname, "r") as f:
        file_content = f.read()
    df = extract_final_table_as_csv(file_content)
    outputs.append(df)
    table_string = data['table'][idx]
    table_string = table_string.replace('<NEWLINE>', '\n')
    table_string_io = StringIO(table_string)
    df_table = pd.read_csv(table_string_io)
    gold.append(df_table)
    print(idx,df.shape)
    
assert(len(outputs) == len(gold))
rmses = []
ers = []

for i in range(0,100):
    rmse, er = compute_rmse_and_er_cell_by_cell_baseline(df_true=gold[i], df_pred=outputs[i])
    rmses.append(rmse)
    ers.append(er)
    print(rmse,er)

avg_rmse = sum(rmses) / len(rmses)
avg_er = sum(ers) / len(ers)

print("Average RMSE over 100 samples:", avg_rmse)
print("Average ER over 100 samples:", avg_er)





0 (2, 9)
1 (2, 9)
2 (2, 9)
3 (2, 9)
4 (2, 9)
5 (2, 9)
6 (2, 9)
7 (2, 9)
8 (2, 9)
9 (2, 9)
10 (2, 9)
11 (2, 9)
12 (2, 9)
13 (2, 9)
14 (2, 9)
15 (2, 9)
16 (2, 9)
17 (2, 9)
18 (2, 9)
19 (2, 9)
20 (2, 9)
21 (2, 9)
22 (2, 9)
23 (2, 9)
24 (2, 9)
25 (2, 9)
26 (2, 9)
27 (2, 9)
28 (2, 9)
29 (2, 9)
30 (2, 9)
31 (2, 9)
32 (2, 9)
33 (2, 9)
34 (2, 9)
35 (2, 9)
36 (2, 9)
37 (2, 9)
38 (2, 9)
39 (2, 9)
40 (2, 9)
41 (2, 9)
42 (2, 9)
43 (2, 9)
44 (2, 9)
45 (2, 9)
46 (2, 9)
47 (2, 9)
48 (2, 9)
49 (2, 9)
50 (2, 9)
51 (2, 9)
52 (2, 9)
53 (2, 9)
54 (2, 9)
55 (2, 9)
56 (2, 9)
57 (2, 9)
58 (2, 9)
59 (2, 9)
60 (2, 9)
61 (2, 9)
62 (2, 9)
63 (2, 9)
64 (2, 9)
65 (2, 9)
66 (2, 9)
67 (2, 9)
68 (2, 9)
69 (2, 9)
70 (2, 9)
71 (2, 9)
72 (2, 9)
73 (2, 9)
74 (2, 9)
75 (2, 9)
76 (2, 9)
77 (2, 9)
78 (2, 9)
79 (2, 9)
80 (2, 9)
81 (2, 9)
82 (2, 9)
83 (2, 9)
84 (2, 9)
85 (2, 9)
86 (2, 9)
87 (2, 9)
88 (2, 9)
89 (2, 9)
90 (2, 9)
91 (2, 9)
92 (2, 9)
93 (2, 9)
94 (2, 9)
95 (2, 9)
96 (2, 9)
97 (2, 9)
98 (2, 9)
99 (2, 9)
2.71569512

/var/folders/13/lbgl_ydd4jlcd4ht05rj1t9r0000gn/T/ipykernel_81095/1589853255.py:64: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_pred = df_pred.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/var/folders/13/lbgl_ydd4jlcd4ht05rj1t9r0000gn/T/ipykernel_81095/1589853255.py:64: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_pred = df_pred.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/var/folders/13/lbgl_ydd4jlcd4ht05rj1t9r0000gn/T/ipykernel_81095/1589853255.py:64: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_pred = df_pred.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/var/folders/13/lbgl_ydd4jlcd4ht05rj1t9r0000gn/T/ipykernel_81095/1589853255.py:64: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_pred = df_pred.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/var/folders/13/lbgl_ydd

****
*****
******

### End to End Eval: TableFill Method

In [2]:
import numpy as np
import pandas as pd

def compute_rmse_and_er_cell_by_cell(df_true: pd.DataFrame, df_pred: pd.DataFrame) -> (float, float):
    # Rename 'Shots Taken' to 'Shots' in the predicted DataFrame if present
    if 'Shots Taken' in df_pred.columns:
        df_pred.rename(columns={'Shots Taken': 'Shots'}, inplace=True)
    
    # Set 'Team' as the index in the gold DataFrame if not already
    if 'Team' in df_true.columns:
        df_true.set_index('Team', inplace=True)
    df_true.index.name = None
    
    # The expected rows in a fixed order
    expected_rows = ["Home Team", "Away Team"]
    
    # Ensure both DataFrames contain at least these two teams
    if not set(expected_rows).issubset(df_true.index):
        raise ValueError("The gold table does not contain both 'Home Team' and 'Away Team'.")
    if not set(expected_rows).issubset(df_pred.index):
        raise ValueError("The predicted table does not contain both 'Home Team' and 'Away Team'.")

    # Reorder rows in both DataFrames to match the expected order
    df_true = df_true.loc[expected_rows]
    df_pred = df_pred.loc[expected_rows]

    # Align df_pred to df_true's columns for a perfect one-to-one cell mapping
    df_pred = df_pred.reindex(columns=df_true.columns)

    # Check shapes after alignment
    if df_true.shape != df_pred.shape:
        raise ValueError("Both tables must have the same shape after alignment.")

    # Ensure that both have the same columns
    if list(df_true.columns) != list(df_pred.columns):
        raise ValueError("Both tables must have the same columns in the same order.")

    # Number of cells
    n = df_true.size
    if n == 0:
        raise ValueError("Tables are empty; no cells to compare.")
    
    # Extract values as numpy arrays
    true_values = df_true.values
    pred_values = df_pred.values
    
    squared_diff_sum = 0.0
    error_count = 0

    # Iterate cell-by-cell
    for r in range(df_true.shape[0]):
        for c in range(df_true.shape[1]):
            gt_val = true_values[r, c]
            pred_val = pred_values[r, c]
            
            diff = gt_val - pred_val
            squared_diff_sum += diff * diff
            
            if gt_val != pred_val:
                error_count += 1

    # Compute metrics
    rmse = np.sqrt(squared_diff_sum / n)
    er = (error_count / n)*100

    return rmse, er

import pandas as pd
import re
from io import StringIO

def extract_final_table(file_contents: str) -> pd.DataFrame:
    # Locate the start of the final table section
    start_match = re.search(r"###\s*Final\s*Table", file_contents)
    if not start_match:
        raise ValueError("No '### Final Table' heading found.")
        
    # Extract the substring starting from ### Final Table
    final_table_start = start_match.end()
    text_after_final_table = file_contents[final_table_start:].strip()
    
    # The table lines each end with '<NEWLINE>' according to the given example.
    # We'll capture all lines up to the last line containing <NEWLINE>.
    # We'll do this by splitting into lines and collecting until there's no more line with <NEWLINE>.
    
    lines = text_after_final_table.split('\n')
    
    # Clean lines (strip whitespace) and ignore empty lines before the table starts
    lines = [line.strip() for line in lines if line.strip()]
    
    # Now find all table lines (those starting with '|' and containing <NEWLINE>)
    table_lines = [line for line in lines if line.startswith('|') and '<NEWLINE>' in line]
    
    if not table_lines:
        raise ValueError("No table lines found in the final table section.")
    
    # The first line is the header line
    header_line = table_lines[0]
    data_lines = table_lines[1:]
    
    # Extract columns from the header line
    # Split by '|', strip spaces, filter out empty and "<NEWLINE>"
    columns = [col.strip() for col in header_line.split('|') if col.strip() and col.strip() != '<NEWLINE>']
    
    # Process data lines similarly
    data = []
    for line in data_lines:
        parts = [part.strip() for part in line.split('|') if part.strip() and part.strip() != '<NEWLINE>']
        # Replace 'Not found' with 0
        parts = [0 if p == 'Not found' else p for p in parts]
        data.append(parts)
    
    # The first column might be empty or a placeholder for the row index
    # In the provided example, columns look like: ['', 'Goals', 'Red Cards', ...]
    # Remove the first empty column header if it exists
    if columns and columns[0] == '':
        columns = columns[1:]
        
    # The first element in each data row is likely the team name. Set it as index.
    team_names = [row[0] for row in data]
    data_values = [row[1:] for row in data]
    
    df = pd.DataFrame(data_values, index=team_names, columns=columns)
    
    # Replace any remaining "Not found" just in case
    df.replace("Not found", 0, inplace=True)
    
    # Convert numeric columns to numeric types
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    
    return df


path = 'data/livesum/test.json'
data = pd.read_json(path)
outputs = []
gold = []
# Example usage:
for idx in range(0,100):
    fname = f"model_outputs/Livesum/GPT4o_TableFill_2step/step2/{idx}.txt"
    with open(fname, "r") as f:
        file_content = f.read()
    df = extract_final_table(file_content)
    outputs.append(df)
    table_string = data['table'][idx]
    table_string = table_string.replace('<NEWLINE>', '\n')
    table_string_io = StringIO(table_string)
    df_table = pd.read_csv(table_string_io)
    gold.append(df_table)
    print(idx,df.shape)
    
assert(len(outputs) == len(gold))
rmses = []
ers = []

for i in range(0,100):
    rmse, er = compute_rmse_and_er_cell_by_cell(df_true=gold[i], df_pred=outputs[i])
    rmses.append(rmse)
    ers.append(er)
    print(rmse,er)

avg_rmse = sum(rmses) / len(rmses)
avg_er = sum(ers) / len(ers)

print("Average RMSE over 100 samples:", avg_rmse)
print("Average ER over 100 samples:", avg_er)



0 (2, 8)
1 (2, 8)
2 (2, 8)
3 (2, 8)
4 (2, 8)
5 (2, 8)
6 (2, 8)
7 (2, 8)
8 (2, 8)
9 (2, 8)
10 (2, 8)
11 (2, 8)
12 (2, 8)
13 (2, 8)
14 (2, 8)
15 (2, 8)
16 (2, 8)
17 (2, 8)
18 (2, 8)
19 (2, 8)
20 (2, 8)
21 (2, 8)
22 (2, 8)
23 (2, 8)
24 (2, 8)
25 (2, 8)
26 (2, 8)
27 (2, 8)
28 (2, 8)
29 (2, 8)
30 (2, 8)
31 (2, 8)
32 (2, 8)
33 (2, 8)
34 (2, 8)
35 (2, 8)
36 (2, 8)
37 (2, 8)
38 (2, 8)
39 (2, 8)
40 (2, 8)
41 (2, 8)
42 (2, 8)
43 (2, 8)
44 (2, 8)
45 (2, 8)
46 (2, 8)
47 (2, 8)
48 (2, 8)
49 (2, 8)
50 (2, 8)
51 (2, 8)
52 (2, 8)
53 (2, 8)
54 (2, 8)
55 (2, 8)
56 (2, 8)
57 (2, 8)
58 (2, 8)
59 (2, 8)
60 (2, 8)
61 (2, 8)
62 (2, 8)
63 (2, 8)
64 (2, 8)
65 (2, 8)
66 (2, 8)
67 (2, 8)
68 (2, 8)
69 (2, 8)
70 (2, 8)
71 (2, 8)
72 (2, 8)
73 (2, 8)
74 (2, 8)
75 (2, 8)
76 (2, 8)
77 (2, 8)
78 (2, 8)
79 (2, 8)
80 (2, 8)
81 (2, 8)
82 (2, 8)
83 (2, 8)
84 (2, 8)
85 (2, 8)
86 (2, 8)
87 (2, 8)
88 (2, 8)
89 (2, 8)
90 (2, 8)
91 (2, 8)
92 (2, 8)
93 (2, 8)
94 (2, 8)
95 (2, 8)
96 (2, 8)
97 (2, 8)
98 (2, 8)
99 (2, 8)
0.35355339

******************************************
**********
*******

In [3]:
import pandas as pd
from io import StringIO
idx = 3
path = 'data/livesum/test.json'
df = pd.read_json(path)
table_string = df['table'][idx]
table_string = table_string.replace('<NEWLINE>', '\n')
table_string_io = StringIO(table_string)
df_table = pd.read_csv(table_string_io)
print(df_table.to_string(index=False))

     Team  Goals  Shots  Fouls  Yellow Cards  Red Cards  Corner Kicks  Free Kicks  Offsides
Away Team      2      8      9             2          0             5          12         5
Home Team      1     11     12             2          0             3           9         4


In [4]:
print(df['text'][idx])

And we're off for the first half. Player26(Away Team) earns a free kick in their defensive half following a foul by Player7(Home Team). Player7(Home Team) receives a yellow card for a rough tackle. The Away Team earns a corner kick. Player27(Away Team)'s shot from the center of the box, assisted by Player20(Away Team), just missed to the left. Player10(Home Team)'s move is risky. Player22(Away Team) earns a free kick in their own half. Player27(Away Team)'s left footed shot from outside the box was blocked with an assist from Player28(Away Team). Player6(Home Team) earns a free kick in their own half. Player27(Away Team) commits a foul. Player8(Home Team)'s left footed shot from outside the box goes wide to the right, with an assist from Player9(Home Team). Player25(Away Team)'s left footed shot from outside the box, assisted by Player28(Away Team), is blocked after the Away Team's corner kick is obtained. Player5(Home Team) is currently sidelined due to an injury, causing a delay in t